# Gedeelde Ontwerperfenis Kwantificeren in een Vermogenstransformatorenpark met PROC INBREED

## Managementsamenvatting

De onderstations-transformatoren van een netbeheerder worden ontworpen in opeenvolgende ontwerpgeneraties, waarbij elk nieuw model wordt afgeleid van twee voorgaande ontwerpen. Deze notebook behandelt die technische stamboom als een pedigree en gebruikt **PROC INBREED** om inteelt- en additieve-verwantschapscoëfficiënten (covariantie) te berekenen, waarmee wordt gekwantificeerd hoeveel ontwerperfenis twee assets delen.

De pedigree is zo opgebouwd dat de structuur interpreteerbaar is: twee van de vier huidige vlootmodellen stammen af van een paar **zuster**-voorgaande ontwerpen en dragen daardoor geconcentreerde erfenis met zich mee, terwijl de andere putten uit gescheiden afstammingslijnen. De procedure herstelt dit exact. De twee uit zustermodellen afgeleide modellen (`G3_T1`, `G3_T3`) hebben elk een inteeltcoëfficiënt van **F = 0,25**; de andere twee (`G3_T2`, `G3_T4`) hebben **F = 0**. De gemiddelde inteeltcoëfficiënt van de vloot is **0,0417**. Bij het aflezen van de additieve-verwantschapsmatrix is het minst verwante paar in de huidige vloot **`G3_T1` & `G3_T3` (verwantschap = 0)** — ze delen geen gemeenschappelijke afstamming en vormen de sterkste redundante (N-1) koppeling, wat van belang is omdat `G3_T1` zelf een van de meest ingeteelde ontwerpen is.

## Gegevensbronnen

| Dataset | Rijen | Kernvariabelen | Beschrijving |
|---------|------|---------------|-------------|
| `DesignLineage` | 12 | `Generation`, `ID`, `Parent1`, `Parent2`, `Sex` | Een kleine, deterministische technische pedigree van onderstations-transformatorontwerpen over drie niet-overlappende ontwerpgeneraties (4 oprichtende platforms, 4 tweede-generatie derivaten, 4 huidige vlootmodellen). Elk niet-oprichtend ontwerp registreert de twee onderscheiden voorgaande ontwerpen (`Parent1`, `Parent2`) waarvan het is afgeleid. `Sex` markeert de leidende ontwerprol (M = mechanische/kernafstamming, F = elektrische/wikkelingsafstamming). De pedigree is handmatig gespecificeerd — niet willekeurig — zodat de verwantschapsstructuur vooraf bekend is en kan worden gecontroleerd aan de hand van de uitvoer van de procedure. |

# Gedeelde Ontwerperfenis Kwantificeren in een Vermogenstransformatorenpark

## Waarom een nutsbedrijf zich druk maakt over "inteelt"

Een transmissie- en distributiebeheerder exploiteert honderden vermogenstransformatoren in onderstations. Om technische kosten en kwalificatie-inspanningen te beheersen, ontwerpen fabrikanten zelden elke transformator vanaf nul — elk nieuw model **erft** kerngeometrie, wikkelingstopologie, isolatiesystemen en doorvoerontwerpen van één of twee voorgaande modellen. Over meerdere aanbestedingscycli ontstaat zo een echte *technische stamboom*: een pedigree van ontwerpen.

Die gedeelde erfenis is een verborgen betrouwbaarheidsrisico. Als twee transformatoren die dezelfde belasting beschermen (een redundant **N-1**-paar) afstammen van hetzelfde voorouderlijke ontwerp, is een latent ontwerpdefect — een resonantiemodus, een isolatieverouderingsmechanisme, een doorslagpad bij de doorvoering — waarschijnlijk aanwezig in **beide** eenheden. Eén hoofdoorzaak kan dan het redundante paar tegelijk uitschakelen: een *common-mode-storing*.

**PROC INBREED** is gebouwd om precies dit soort gedeelde afstamming te kwantificeren. Hoewel de oorsprong ervan in de dieren- en plantenveredeling ligt, is de wiskunde — Wrights recursie voor additieve verwantschap — domeinonafhankelijk: ze meet het aandeel ontwerperfenis dat twee individuen delen via gemeenschappelijke voorouders. We vertalen de genetica-terminologie naar asset engineering:

| INBREED-concept | Interpretatie voor het nutsbedrijf |
|---|---|
| Individu | Een transformatorontwerp / -model |
| Ouder (vader/moeder) | Een voorgaand ontwerp waarvan het is afgeleid |
| Generatie (CLASS) | Een aanbestedings-/ontwerpcyclus |
| Inteeltcoëfficiënt *F* | Mate van zelfgelijkende erfenis binnen een ontwerp |
| Covariantie-/verwantschapscoëfficiënt | Gedeelde ontwerperfenis tussen twee ontwerpen |
| Minst verwant paar | Beste redundante koppeling voor N-1-veerkracht |

## Stap 1 — Specificeer de ontwerppedigree

We voeren `DesignLineage` rechtstreeks in zodat de verwantschapsstructuur ondubbelzinnig is. Er zijn drie **niet-overlappende ontwerpgeneraties**:

- **Generatie 1** — vier oprichtende platformontwerpen (`G1_P1`-`G1_P4`) zonder voorgangers (lege ouders).
- **Generatie 2** — vier derivatenontwerpen (`G2_D1`-`G2_D4`), elk ontwikkeld uit twee **onderscheiden** Generatie-1-platforms. `G2_D1` en `G2_D2` zijn *volle zustermodellen* (beide uit `G1_P1` x `G1_P2`); `G2_D3` en `G2_D4` zijn volle zustermodellen (beide uit `G1_P3` x `G1_P4`).
- **Generatie 3** — vier huidige vlootmodellen (`G3_T1`-`G3_T4`). `G3_T1` is opgebouwd uit het zusterpaar `G2_D1` x `G2_D2`, en `G3_T3` uit het zusterpaar `G2_D3` x `G2_D4`; deze twee ontwerpen concentreren daarom erfenis. `G3_T2` en `G3_T4` kruisen elk de twee gescheiden afstammingslijnen.

De kolommen `ID`, `Parent1` en `Parent2` dragen de pedigree; `Sex` registreert welke technische afstammingslijn het ontwerp leidde. Een korte vervolg-DATA-stap maakt de plaatshouder `.` leeg, zodat de oprichtende platforms lege ouders hebben, zoals PROC INBREED verwacht.

In [1]:
GEGEVENS DesignLineage;
   LENGTE ID $12 Parent1 $12 Parent2 $12 Sex $1;
   INFILE DATALINES truncover;
   INVOER Generation ID $ Parent1 $ Parent2 $ Sex $;
   DATALINES;
1 G1_P1 . . M
1 G1_P2 . . M
1 G1_P3 . . M
1 G1_P4 . . F
2 G2_D1 G1_P1 G1_P2 M
2 G2_D2 G1_P1 G1_P2 F
2 G2_D3 G1_P3 G1_P4 F
2 G2_D4 G1_P3 G1_P4 M
3 G3_T1 G2_D1 G2_D2 M
3 G3_T2 G2_D1 G2_D3 F
3 G3_T3 G2_D3 G2_D4 F
3 G3_T4 G2_D2 G2_D4 M
;
UITVOEREN;

/* Oprichtende platforms hebben geen voorgangers: maak de plaatshouder-punten leeg */
GEGEVENS DesignLineage;
   INSTELLEN DesignLineage;
   ALS Parent1 = '.' DAN Parent1 = ' ';
   ALS Parent2 = '.' DAN Parent2 = ' ';
UITVOEREN;

TITEL 'Transformatorontwerp-pedigree';
PROCEDURE AFDRUKKEN GEGEVENS=DesignLineage noobs label;
   VARIABELE Generation ID Parent1 Parent2 Sex;
   label Generation="Generatie" ID="ID" Parent1="Ouder 1" Parent2="Ouder 2" Sex="Geslacht (kernlijn)";
UITVOEREN;


                                             Transformatorontwerp-pedigree                                              


Generatie     ID  Ouder 1  Ouder 2  Geslacht (kernlijn)
---------  -----  -------  -------  -------------------
        1  G1_P1                    M
        1  G1_P2                    M
        1  G1_P3                    M
        1  G1_P4                    F
        2  G2_D1  G1_P1    G1_P2    M
        2  G2_D2  G1_P1    G1_P2    F
        2  G2_D3  G1_P3    G1_P4    F
        2  G2_D4  G1_P3    G1_P4    M
        3  G3_T1  G2_D1    G2_D2    M
        3  G3_T2  G2_D1    G2_D3    F
        3  G3_T3  G2_D3    G2_D4    F
        3  G3_T4  G2_D2    G2_D4    M




NOTE: DATA DesignLineage

NOTE: Processing inline DATALINES (12 lines)

NOTE: Read 12 rows from DATALINES.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA DesignLineage


NOTE: Read 12 rows from DesignLineage.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: Option TITLE changed to Transformatorontwerp-pedigree.
NOTE: PROC PRINT data=DesignLineage

NOTE: PROC PRINT completed: 12 observations printed, 5 variables


## Stap 2 — Inteeltcoëfficiënten per ontwerpgeneratie

We voeren PROC INBREED uit in **multi-generatie-modus** door `Generation` in de `CLASS`-instructie op te nemen, wat de pedigree-samenvatting per generatie afdrukt. De `VAR`-instructie levert de drie pedigree-kolommen in de vereiste volgorde: individu, eerste voorganger, tweede voorganger.

- De optie **IND** drukt de inteeltcoëfficiënt *F* van elk ontwerp af — een maat voor hoe geconcentreerd de eigen erfenis is. Een ontwerp dat is opgebouwd uit twee nauw verwante voorgangers heeft een hoge *F*.
- De optie **MATRIX** drukt de volledige additieve-verwantschapsmatrix af, zodat we paarsgewijze erfenis rechtstreeks kunnen aflezen.
- De optie **AVERAGE** voegt de gemiddelde inteeltcoëfficiënt van de hele vloot toe — een enkele samenvatting van ontwerpdiversiteit.

Waarden dicht bij 0 betekenen onafhankelijke afstammingslijnen; *F* stijgt naarmate de twee voorgangers van een ontwerp nauwer verwant raken.

In [2]:
TITEL 'Inteeltcoëfficiënten per Ontwerpgeneratie';
PROCEDURE inbreed GEGEVENS=DesignLineage ind average MATRIX;
   VARIABELE ID Parent1 Parent2;
   KLASSE Generation;
UITVOEREN;


                                       Inteeltcoëfficiënten per Ontwerpgeneratie                                        


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
ID                  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficients (Generation 2)
--------------------------------------
ID                  F (Inbreeding)
G2_D1                 0.000000
G2_D2                 0.000000
G2_D3                 0.000000
G2_D4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coe


NOTE: Option TITLE changed to Inteeltcoëfficiënten per Ontwerpgeneratie.
NOTE: PROC INBREED data=DesignLineage



## Stap 3 — Covariantiecoëfficiënten opgeslagen voor risicoscoring in een vervolgstap

Door de inteeltweergave te vervangen door de optie **COVAR** worden dezelfde relaties gerapporteerd als **covariantie- (additieve-verwantschaps)coëfficiënten**, de vorm die een asset-managementteam zou invoeren in een redundantierisicoscore. De optie **OUTCOV=** legt de volledige matrix vast als een dataset (`DesignCovar`), waarbij de kolommen `Col1`-`Col12` de verwantschap van elk ontwerp met elk ander ontwerp bevatten (in pedigree-volgorde) — klaar om terug te koppelen aan onderstationstoewijzingen.

We drukken de uitvoerdataset af zodat de opgeslagen matrix zichtbaar is naast de lijst.

In [3]:
TITEL 'Covariantie- (Verwantschaps)coëfficiënten';
PROCEDURE inbreed GEGEVENS=DesignLineage covar MATRIX OUTCOV=DesignCovar;
   VARIABELE ID Parent1 Parent2;
   KLASSE Generation;
UITVOEREN;

TITEL 'OUTCOV= Verwantschapsmatrix Opgeslagen voor Risicoscoring';
PROCEDURE AFDRUKKEN GEGEVENS=DesignCovar noobs;
UITVOEREN;


                                       Covariantie- (Verwantschaps)coëfficiënten                                        


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
ID                  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000

Covariance Matrix (Additive Genetic Relationship) (Generation 1)
--------------------------------------------------
ID                     G1_P1    G1_P2    G1_P3    G1_P4
G1_P1                1.0000   0.0000   0.0000   0.0000
G1_P2                0.0000   1.0000   0.0000   0.0000
G1_P3                0.0000   0.0000   1.0000   0.00


NOTE: Option TITLE changed to Covariantie- (Verwantschaps)coëfficiënten.
NOTE: PROC INBREED data=DesignLineage

NOTE: OUTCOV= dataset DesignCovar written.
NOTE: Option TITLE changed to OUTCOV= Verwantschapsmatrix Opgeslagen voor Risicoscoring.
NOTE: PROC PRINT data=DesignCovar

NOTE: PROC PRINT completed: 12 observations printed, 17 variables


## Stap 4 — Minst verwante koppelingen voor redundante (N-1) installaties

De opgeslagen `DesignCovar`-matrix is precies wat een redundantiestudie nodig heeft. De verwantschap van ontwerp *i* met ontwerp *j* staat in rij *i*, kolom `Col`*j* (kolommen staan in pedigree-volgorde, dus `Col9`-`Col12` komen overeen met de vier huidige vlootmodellen `G3_T1`-`G3_T4`).

We lezen de vier rijen van de huidige vloot uit `DesignCovar`, vormen elk ongeordend paar uit de huidige vloot, koppelen de verwantschapscoëfficiënt eraan, en sorteren van minst naar meest verwant. Het doel is redundante paren te kiezen waarvan de gedeelde erfenis **het laagst** is — deze minimaliseren de kans dat één overgeërfd ontwerpgebrek beide helften van een N-1-beschermde positie uitschakelt.

In [4]:
/* Haal de vier huidige-vlootrijen op; Col9..Col12 zijn de verwantschappen
   met G3_T1..G3_T4 (pedigree-kolomvolgorde). Vorm elk ongeordend paar. */
GEGEVENS Pairs;
   INSTELLEN DesignCovar;
   WAAR ID in ('G3_T1','G3_T2','G3_T3','G3_T4');
   LENGTE DesignA $12 DesignB $12;
   REEKS g3{4} Col9 Col10 Col11 Col12;
   REEKS nm{4} $12 _temporary_
      ('G3_T1','G3_T2','G3_T3','G3_T4');
   DesignA = ID;
   DOE j = 1 TOT 4;
      DesignB = nm{j};
      ALS DesignA < DesignB DAN DOE;
         Relationship = g3{j};
         UITVOER;
      EINDE;
   EINDE;
   BEWAREN DesignA DesignB Relationship;
UITVOEREN;

PROCEDURE SORTEREN GEGEVENS=Pairs; VOLGENS Relationship; UITVOEREN;

TITEL 'Kandidaat Redundante (N-1) Koppelingen, Minst Verwant Eerst';
PROCEDURE AFDRUKKEN GEGEVENS=Pairs noobs label;
   VARIABELE DesignA DesignB Relationship;
   label DesignA="Ontwerp A" DesignB="Ontwerp B" Relationship="Verwantschap";
UITVOEREN;
TITEL;


                              Kandidaat Redundante (N-1) Koppelingen, Minst Verwant Eerst                               


Ontwerp A  Ontwerp B  Verwantschap
---------  ---------  ------------
G3_T1      G3_T3                 0
G3_T2      G3_T4              0.25
G3_T1      G3_T2             0.375
G3_T1      G3_T4             0.375
G3_T2      G3_T3             0.375
G3_T3      G3_T4             0.375




NOTE: DATA Pairs


NOTE: Read 12 rows from DesignCovar.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=Pairs

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 6 rows from Pairs.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: Option TITLE changed to Kandidaat Redundante (N-1) Koppelingen, Minst Verwant Eerst.
NOTE: PROC PRINT data=Pairs

NOTE: PROC PRINT completed: 6 observations printed, 3 variables


## Interpretatie van de resultaten

**Inteeltcoëfficiënten (Stap 2).** Alle oprichtende Generatie-1-platforms en alle Generatie-2-derivaten tonen *F* = 0 — door constructie heeft geen enkele twee verwante voorgangers. Twee Generatie-3-modellen komen naar voren met *F* = 0,25: `G3_T1` (opgebouwd uit het zusterpaar `G2_D1` x `G2_D2`) en `G3_T3` (uit het zusterpaar `G2_D3` x `G2_D4`). Hun voorgangers herleiden tot *hetzelfde* paar oprichtende platforms, dus hun erfenis is geconcentreerd. Vanuit betrouwbaarheidsoogpunt zijn dit de ontwerpen die het meest blootstaan aan één overgeërfd defect, en ze verdienen extra onafhankelijke kwalificatietests. `G3_T2` en `G3_T4`, die de twee gescheiden afstammingslijnen kruisen, hebben *F* = 0.

**Verwantschapsmatrix (Stap 3).** De niet-diagonale waarden kwantificeren paarsgewijze gedeelde erfenis. De twee zusterparen van Generatie 2 tonen elk een onderlinge verwantschap van 0,50 (`G2_D1`-`G2_D2` en `G2_D3`-`G2_D4`), terwijl derivaten uit tegengestelde afstammingslijnen op 0,00 staan. De ingeteelde huidige-vlootontwerpen hebben zelfverwantschappen van 1,25 (= 1 + *F*), zichtbaar op de diagonaal. De dataset `DesignCovar` maakt de volledige matrix beschikbaar om te koppelen aan onderstationstoewijzingen.

**Minst verwante koppelingen (Stap 4).** Door elk paar uit de huidige vloot te rangschikken op verwantschap komt **`G3_T1` & `G3_T3` op de eerste plaats met een verwantschap van 0,00** — ze stammen af van volledig gescheiden voorouderlijke platforms en delen geen ontwerperfenis. Dit is de sterkste redundante koppeling, en dit is bijzonder waardevol omdat `G3_T1` zelf een van de twee meest ingeteelde ontwerpen is: het koppelen ervan aan een volledig onverwante partner compenseert de geconcentreerde erfenis. Het op één na beste paar is `G3_T2` & `G3_T4` met 0,25; de overige paren staan op 0,375. De gemiddelde inteeltcoëfficiënt van de vloot van **0,0417** (afgedrukt door de optie AVERAGE in Stap 2) vat de algehele ontwerpdiversiteit samen. Inkoop zou de voorkeur moeten geven aan de paren met de laagste verwantschap voor N-1-posities en, na verloop van tijd, de voorouderlijke basis moeten verbreden — het asset-engineering-equivalent van het vermijden van een genetische bottleneck.

**Kanttekening.** Dit zijn illustratieve synthetische gegevens; een productiestudie zou de pedigree opbouwen uit de werkelijke ontwerprevisiegegevens van de fabrikant en de verwantschapsscores valideren aan de hand van historische common-mode-uitvalgebeurtenissen voordat inkoopbeslissingen worden gestuurd.